<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/Seq2seq%E1%84%8B%E1%85%AA_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seq2seq와 Attention

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 1. Seq2seq와 Attention

![](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/8afab9a7-9614-42f1-88e0-bd12c8cbd2b0.png)

문장을 토큰화하고 의미를 부여하는 과정은 멋지고 중요한 일이지만 조금은 지루할 수 있죠. 그 과정을 견디고 여기까지 오신 여러분, 고생하셨습니다!

이번 코스에서는 언어 모델이 발전해 온 과정에 대해 개략적으로 공부하고, NLP의 큰 흐름 중 하나인 **Sequence to Sequence(Seq2seq)** 에 대해 살펴볼 것입니다. 이를 발전시키기 위한 기법이자 지금은 없어선 안 될 중요한 메커니즘인 **Attention**에 대해서도 최초의 아이디어가 생겨난 시점에서부터 자세히 살펴보려고 합니다. 부디 즐거운 시간이 되시길 바랍니다!

> **[문제] [아이스브레이킹] 문장을 토큰화하고 의미를 부여하는 작업이 무엇인지 기억나시나요?**

## 학습 내용
---
- **2. 우리가 만드는 언어 모델**
    - 언어 모델을 학습합니다.

- **3. Sequence to Sequence 문제**
    - seq2seq 문제가 무엇인지 학습합니다.

- **4. Sequence to Sequence 구현**
    - seq2seq를 Pytorch로 구현합니다.

- **5. Attention!**
    - (1) Bahdanau Attention  
        - Bahdanau Attention을 학습합니다.
    - (2) Luong Attention  
        - Luong Attention을 학습합니다.

- **6. 트랜스포머로 가기 전 징검다리?**
    - 트랜스포머가 무엇일지 생각합니다.

## 학습 목표
---
- 언어 모델이 발전해 온 과정을 개략적으로 설명할 수 있다.
- 기존 RNN 기법이 번역에서 보인 한계를 파악하고, 이를 개선한 Seq2seq를 이해할 수 있다.
- Seq2seq를 발전시킨 Attention에 대해 설명할 수 있다.

## 2. 우리가 만드는 언어 모델

언어 모델(Language Model)이란, 주어진 단어들을 보고 다음 단어를 맞추는 모델입니다. 더 자세하게는, **단어의 시퀀스를 보고 다음 단어에 확률을 할당** 하는 모델이죠!

좀 더 수학적으로 표현하자면, 언어 모델은 n-1개의 단어 시퀀스 $\text{w}_{1}, \cdots, \text{w}_{n-1}$가 주어졌을 때, n번째 단어 $\text{w}_{n}$으로 무엇이 올지를 예측하는 확률 모델로 표현됩니다. 파라미터 $\theta$로 모델링하는 언어 모델을 다음과 같이 표현할 수 있습니다.

$$\text{P}(\text{w}_{n}|\text{w}_{1},\cdots,\text{w}_{n-1};\theta)$$

하지만 나중에는 꼭 시퀀스 형태의 Next Token Prediction 언어모델이 아니더라도, 주변 단어를 보고 중심 단어를 예측하는 형태로 언어모델을 구성하는 것도 보시게 될 것입니다.

## 통계적 언어 모델 (Statistical Language Model)
---
딥러닝이 등장하기 이전엔 **통계적 언어 모델(Statistical Language Model)** 의 사용이 지배적이었습니다. 대표적으로 2000년대 초반까지 구글이나 네이버의 번역기는 모두 통계적 언어 모델을 기반으로 하고 있었죠. 통계적 언어 모델의 이모저모를 배우기 위해 아래 웹페이지를 방문해 보세요.

- [언어모델(Language Model)](https://wikidocs.net/21687)

> **[문제] 위 글에서 설명하는 언어 모델은 충분한 데이터가 없다면 범용적인 모델을 구축하기 어렵습니다. 그 이유에 대해 간단히 적어보세요.**

`등장한 적 없는 단어나 문장에 대해 모델링을 할 수 없다`는 통계적 언어 모델의 단점은 치명적이었습니다. 데이터가 아무리 많다고 해도 세상 모든 단어를 포함할 수는 없었으니 말이죠.

## 신경망 언어 모델 (Neural Network Language Model)
---
통계적 언어 모델의 단점을 개선한 것이 우리가 배울 **신경망 언어 모델(Neural Network Language Model, 이하 NNLM)** 입니다. NNLM의 시초는 Feed-Forward 신경망 언어 모델인데, 지금의 Embedding 레이어의 아이디어인 모델입니다. 이에 대한 설명이 아래 웹페이지에 아주 잘 되어 있습니다.

- [피드 포워드 신경망 언어 모델(NNLM)](https://wikidocs.net/45609)

> **[문제] 위 링크의 글에서 설명하는 희소문제(sparsity problem)에 대해 간단히 설명해 보세요.**

> **[문제] 출력 층에서 사용되는 활성함수 Softmax는 어떤 기능적 의미를 갖나요?**

단어를 어떤 Embedding으로 표현할 수 있다는 것은 어떤 이점을 가질까요? 단순 유사도 비교를 넘어서는 활용 사례를 아래 영상에서 확인해보세요!

### 🎬

[영상 보기](https://youtu.be/gUMvBRI-WGo)

각 단어를 일련의 Embedding 벡터로 표현한 후, 이전의 몇 개 단어를 활용해 다음 단어를 예측하는 것은 분명 많은 문제를 해결했습니다. 특히 `단어 간의 유사도를 표현할 수 있게 되어` 문장의 유창성이 높아진 것은 혁신적이었죠!

하지만 예측에 `정해진 개수의 단어만 참고한다`는 분명한 한계가 있었습니다. 예를 들어 번역문을 생성하려면 문장이 짧을 수도, 길 수도 있으니, n개의 단어를 참고하기보다는 "몇 개 단어가 들어와도 문장 단위로 처리한다!"는 종류의 모델링이 필요하게 되었죠. 그렇게 고안된 것이 바로 여러분들이 잘 알고 계신 **순환 신경망(Recurrent Neural Network, 이하 RNN)을 활용한 언어 모델** 입니다. 이에 대한 설명은 다음 스텝에서 이어갈게요!

## 3. Sequence to Sequence 문제

![https://wikidocs.net/45609](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/76fc5964-4ef9-4b67-89ce-3e74c7ff6748.png)

NNLM의 구조

다시 한번 설명하면, 여러 개의 단어(Embedding)를 합쳐(Concatenate) 고정된 크기의 Weight를 Linear로 처리하는 방식은 유연성에 한계가 있었습니다. 단어의 개수에 무관하게 처리할 수 있는 네트워크가 필요했고, 그것은 곧 **RNN의 고안**으로 이어졌습니다. RNN은 고정된 크기의 Weight가 선언되는 것은 동일하지만 입력을 **순차적으로 "적립"하는 방식**을 채택함으로써 유동적인 크기의 입력을 처리할 수 있었습니다.

"**적립**"이라는 표현을 일반적으로 사용하지는 않습니다만, 필자가 생각하기에 RNN의 동작 방식에 잘 어울린다고 생각해 선택했습니다. 기억, 누적, 압축 등으로 대체할 수 있으며 가장 와닿는 방향으로 이해하세요!

![https://towardsdatascience.com/illustrated-guide-to-recurrent-neural-networks-79e5eb8049c9](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/60c9330e-e738-4dcb-9902-0b5cd6ac41fd.png)

단어가 자체적으로 의미를 가질 수 있는 Embedding을 도입하고, 입력의 유연성을 위해 RNN도 적용했는데, 아직도 해결할 문제가 있을까요? 물론입니다! 대표적으로 RNN에는 **두 가지 문제점**이 꼽히곤 합니다.

1. 하나의 Weight에 입력을 적립하다 보니 입력이 길어질수록 이전 입력에 대한 정보가 소실되는 **기울기 소실(Vanishing Gradient)** 문제가 있습니다. 위의 그림을 보면 각 입력마다 정보를 색으로 확인할 수 있습니다. 첫 입력인 `What(남색)`의 정보가 마지막 입력인 `?`에 다다라서는 **거의 희석된 모습**을 보여주고 있죠. 이 문제는 LSTM을 고안함으로써 개선되었습니다. LSTM에 대한 자세한 내용은 이미 알고 계신다고 생각하겠습니다. 기억이 잘 나지 않는 분은 아래 링크를 참고해 주세요.

- [Long Short-Term Memory (LSTM) 이해하기](https://dgkim5360.tistory.com/entry/understanding-long-short-term-memory-lstm-kr)

2. 단어 단위로 입력과 출력을 순환하는 RNN 구조는 문장 생성엔 적합할지언정 **번역에 사용하기는 어렵다는 문제가 있습니다**. `나는 점심을 먹는다` 라는 문장을 영문으로 번역하자면 목표 문장은 `I eat lunch` 가 될 것인데, 과정을 순차적으로 생각하면 `eat` 이라는 단어를 만들 때는 먹는다 에 대한 정보가 없습니다. 각 언어별로 어순이 다르기 때문입니다.

> 나는 -> I
>
> 나는 점심을 -> I lunch
>
> 나는 점심을 먹는다 -> I lunch eat(?)

심지어 입력의 길이와 번역의 길이가 같다는 보장도 없죠. 번역에 있어서는 문장을 다 읽고 번역하는, 즉 **문장 전체를 보고 나서 생성하는 구조**가 필요했습니다. 이에 2014년, 구글이 *Sequence to Sequence(Seq2Seq)* 구조를 제안합니다.

- 논문: [Sequence to Sequence Learning with Neural Networks](https://papers.nips.cc/paper/5346-sequence-to-sequence-learning-with-neural-networks.pdf)

![https://papers.nips.cc/paper/5346-sequence-to-sequence-learning-with-neural-networks.pdf](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/c400f678-109b-4de8-86aa-b6f6cd836f95.png)

Sequence to Sequence Learning with Neural Networks

이를 멋지게 정리한 글을 함께 첨부합니다.

- [Seq2seq (2): Sequence to Sequence Learning with Neural Networks](https://reniew.github.io/35/)

> **[문제] seq2seq의 저자들은 단순한 RNN 대신 LSTM을 사용하고자 하였습니다. 그 이유를 간단히 서술하고 논문에서 해당 내용이 포함된 문단을 찾아보세요.**

**<식 1>**

$$p\left(y_1, \ldots, y_{T^{\prime}} \mid x_1, \ldots, x_T\right)=\Pi_{t=1}^{T^{\prime}} p\left(y_t \mid v, y_1, \ldots, y_{t-1}\right)$$

> **[문제] 위의 <식 1>을 보고 각 x와 y, v가 무엇을 의미하는지 설명해볼까요? 그리고 v의 정의를 논문에서 찾아 적어보세요.**

윗글은 논문의 내용에 포커싱이 되어있다면 아랫글은 작동 방식에 대한 조금 더 자세한 설명을 포함합니다. 윗글을 잘 이해했다면 아랫글을 읽으며 복습하고 논문에는 언급되어 있지 않은 디테일한 내용을 파악하도록 합시다. 글 내용 중 `1. 시퀀스-투-시퀀스(Sequence-to-Sequence)` 부분을 읽어보세요!

- [시퀀스-투-시퀀스](https://wikidocs.net/24996)

> **[문제] 문장의 시작과 끝에 붙는 특수한 토큰들이 있습니다. 만약 그 토큰들이 없다면 어떤 상황이 벌어질까요? 시작 토큰이 없는 경우와 끝 토큰이 없는 경우를 각각 생각해봅시다.**

## 4. Sequence to Sequence 구현

이제 Sequence to Sequence를 `Pytorch`로 구현해보죠. 일단은 데이터를 직접 다루기보다는 차원 수를 확인하는 실습을 해보겠습니다. RNN 계통의 레이어들은 입력값과 반환값이 설정에 따라 각양각색입니다. 이번 구현에서는 입력으로 **Embedding된 단어만 전달**하고 (Hidden State는 전달하지 않습니다), 출력은 Encoder와 Decoder 별로 상이하므로 각각 설명을 첨부하겠습니다.

## LSTM Encoder
---

In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        print("입력 Shape:", src.size())

        embedded = self.embedding(src)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        outputs, (h_0, c_0) = self.rnn(embedded)
        print("LSTM Layer의 Output Shape:", outputs.size())
        print("LSTM Layer의 Hidden State Shape:", h_0.size())
        print("LSTM Layer의 Cell State Shape:", c_0.size())

        return outputs, h_0, c_0

print("슝~")

슝~


Embedding 레이어를 단어 사이즈와 Embedding 차원에 대해 선언을 한 후, 논문에서 소개한 대로 `torch.nn.LSTM(emb_dim, hidden_dim, batch_first=True)`으로 LSTM을 정의합니다. PyTorch 의 LSTM 은 항상 두 가지를 돌려줍니다 — 모든 스텝의 출력 `outputs` 와 마지막 스텝의 상태 `(h, c)`. Keras 처럼 `return_sequences` 를 켤 필요가 없어요. 이 중 마지막 상태 `(h, c)` 가 논문의 **컨텍스트 벡터(Context Vector)** 역할을 합니다. 추가적인 옵션이 궁금하시다면 아래의 Pytorch LSTM 공식 문서를 참조하시면 좋습니다.

- [torch.nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

In [ ]:
vocab_size = 30000
emb_size = 256
lstm_size = 512
batch_size = 1
sample_seq_len = 3

print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



In [ ]:
import torch

encoder = Encoder(vocab_size, emb_size, lstm_size)
sample_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

sample_output, hidden, cell = encoder(sample_input)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
LSTM Layer의 Hidden State Shape: torch.Size([1, 1, 512])
LSTM Layer의 Cell State Shape: torch.Size([1, 1, 512])


![예시 코드와 동일한 Shape를 가지는 Encoder 구조](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/496ca52e-561c-47d8-9c65-b0973215574e.png)

아주 간편하게 `Encoder` 클래스를 정의했습니다. 어떤 Source 문장을 `Encoder`에 읽히고, 그 반환 값인 LSTM의 최종 State 값을 `Decoder`에게 전달해 주면 되겠죠?

## LSTM Decoder
---

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, context):
        print("입력 Shape:", x.size())

        embedded = self.embedding(x)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        embedded = torch.cat((embedded, context), dim=2)
        print("Context Vector가 더해진 Shape:", embedded.size())

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        print("LSTM Layer의 Output Shape:", output.size())

        output = self.fc(output)
        print("Decoder 최종 Output Shape:", output.size())

        return output, hidden, cell

`Decoder`는 `Encoder`와 구조적으로 유사하지만 결과물을 생성해야 하므로 Fully Connected 레이어(`self.fc`)가 추가되어 단어장 크기의 점수를 냅니다. Softmax 는 모델 안에 두지 않고 학습 때 `CrossEntropyLoss` 가 내부에서 적용합니다. PyTorch LSTM 은 기본으로 모든 스텝의 출력을 돌려주므로, 매 스텝의 번역 결과에 해당하는 `output` 시퀀스를 그대로 `fc` 에 넘깁니다.

In [ ]:
print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



In [ ]:
decoder_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))
decoder = Decoder(vocab_size, emb_size, lstm_size)

# 인코더 최종 상태를 디코더 매 스텝에 같은 값으로 붙임
context = hidden.transpose(0, 1).expand(-1, sample_seq_len, -1)

dec_output, hidden, cell = decoder(decoder_input, hidden, cell, context)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
Context Vector가 더해진 Shape: torch.Size([1, 3, 768])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
Decoder 최종 Output Shape: torch.Size([1, 3, 30000])


![예시 코드와 동일한 Shape를 가지는 Seq2seq 구조](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/0f8e94df-5ba4-4c84-b9c2-0155538b56d1.png)

이 수식을 기억하시나요?

$$p\left(y_1, \ldots, y_{T^{\prime}} \mid x_1, \ldots, x_T\right)=\Pi_{t=1}^{T^{\prime}} p\left(y_t \mid v, y_1, \ldots, y_{t-1}\right)$$

`Encoder`가 생성한 컨텍스트 벡터 v 를 Embedding 레이어를 거친 y 값에 Concatnate하여 위 수식을 비로소 만족하게 됩니다. 우리가 Seq2seq를 완성한 거죠!

## 5. Attention!

![](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/797fc238-1072-4f1c-a32d-231bf3ff9b78.png)

혁신적이었던 Seq2Seq은 Encoder-Decoder 구조라는 딥러닝 모델의 큰 틀을 제시했고, 지금까지도 그 구조는 널리 활용되고 있습니다. 하지만 그것만으로 기계 번역이 완벽했다면 우리는 이미 외국어에 대한 두려움이 사라졌겠죠?

Seq2Seq 역시 여느 기법처럼 한계점이 존재했으며 이를 발전시키려는 시도가 아주 많았습니다. 가장 대표적인 방법이 바로 **Attention 메커니즘**입니다! 이번 스텝에서는 Attention 메커니즘의 이모저모를 살펴볼 거예요!

## (1) Bahdanau Attention
---

장점으로 보이는 어떤 것도 누군가에게는 결점이 보이는 법이죠. Bahdanau는 Seq2Seq의 **컨텍스트 벡터가 고정된 길이로 정보를 압축하는 것이 손실을 야기한다**고 주장하였습니다. 즉 짧은 문장에 대해서는 괜찮을지 모르겠으나 문장이 길어질수록 성능이 저하된다는 것이지요. 콜럼버스의 달걀처럼, 듣고 보니 그럴듯합니다!

이에 그는 Encoder의 최종 State 값만을 사용하는 기존의 방식이 아닌, 매 스텝의 Hidden State를 활용해 컨텍스트 벡터를 구축하는 **Attention 메커니즘**을 제안합니다.

아래는 Bahdanau의 Attention 논문입니다. 내용이 제법 어렵지만 must-read 논문이니만큼, 꼭 한번 읽어보시기를 권합니다. 하지만 지금은 소개해 드릴 정리 글과 시각화 자료를 통해 조금 더 쉽게 이해해보세요!

- 원본 논문: [NEURAL MACHINE TRANSLATION BY JOINTLY LEARNING TO ALIGN AND TRANSLATE](https://arxiv.org/pdf/1409.0473.pdf)

- 정리 글: [Attention mechanism in NLP. From seq2seq + attention to BERT](https://lovit.github.io/machine%20learning/2019/03/17/attention_in_nlp/) (지금까지 배운 것을 확인하기에도 아주 좋습니다!)

> **[문제] 블로그 저자에 따르면 모델의 성능 향상 외에 Attention을 유용하게 사용할 수 있는 방법이 하나 있습니다. 부산물이라고 표현된 그 방법은 무엇인가요?**

### seq2seq과 attn-seq2seq, 뭐가 다른가?
---
Attention의 개념에 대해 어려워하는 분들이 많습니다. 이렇게 생각해 봅시다. Attention이 없는 것과 있는 것은 과연 뭐가 달라지는 것일까?

- seq2seq
$$p\left(y_i \mid y_1, \ldots, y_{i-1}, \mathbf{x}\right)=g\left(y_{i-1}, s_i, c\right)$$

- attention
$$p\left(y_i \mid y_1, \ldots, y_{i-1}, \mathbf{x}\right)=g\left(y_{i-1}, s_i, c_{i}\right)$$

위의 두 식을 비교해 봅시다. Bahdanau 논문 원문에 나오는 Encoder-Decoder 구조에 대한 수식을 seq2seq만 있는 경우와 attention이 적용된 경우로 나누어 비교해 보면, 약간의 notation을 수정해서 보면 단 한 군데만 빼고는 사실상 동일합니다. 어디가 다른지 눈에 띄시나요?

네 그렇습니다. 유일한 차이는 attention이 있는 경우엔 바로 context vector c에 첨자 i가 붙어있다는 점입니다. 그렇다면 이 작은 첨자 하나가 어떤 근본적인 차이를 가져오게 되는 것인지 생각해 볼까요?

![](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/81f11a8e-cc52-4f15-b06d-e3ac2b8e48ec.png)

위 그림은 Bahdanau 논문 원문의 3.LEARNING TO ALIGN AND TRANSLATE의 내용을 바탕으로 재구성한 것입니다. 위 그림의 왼쪽 부분은 $\textbf{X}_{j}$를 입력으로, $\textbf{y}_{i}$를 출력으로 하는 인코더-디코더 부분을 도식화한 것입니다. 여기서 유의해야 할 점은 **$i$는 디코더의 인덱스, $j$는 인코더의 인덱스**라는 점입니다.

그렇다면 context vector c에 첨자 i가 붙어 $c_{i}$가 된다는 것의 의미는 무엇일까요?

>인코더가 X를 해석한 context $c_{i}$는 디코더의 포지션 i에 따라 다르게 표현(represent)되어야 한다.

seq2seq의 인코더가 해석한 context는 디코더의 포지션 i에 무관하게 항상 일정했습니다. 그러나 attention이 가미되면 달라집니다.

'나는 밥을 먹었다'라는 한글 문장을 'I ate lunch'로 번역한다고 생각해 봅시다. 영어 문장의 첫 번째(i=0) 단어 'I'를 만들어야 할 때 인코더가 한글 문장을 해석한 컨텍스트 벡터에서는 '나는'이 강조되어야 하고, 영어 문장의 세 번째(i=2) 단어 'lunch'를 만들어야 할 때 인코더의 컨텍스트 벡터에서는 '밥을'이 강조되어야 한다는 것입니다. **디코더가 현재 시점 i에서 보기에 인코더의 어느 부분 j가 중요한가?** 이 가중치가 바로 attention인 것입니다.

얼마나 강조되어야 하는지를 나타내는 가중치는 어떻게 계산하나요? 위의 식에서 $\alpha_{ij}$가 바로 인코더의 j번째 hidden state $h_{j}$가 얼마나 강조되어야 할지를 결정하는 가중치 역할을 합니다. 이 가중치는 다시 디코더의 직전 스텝의 hidden state $s_{i-1}$와 $h_{j}$의 유사도가 높을수록 높아지게 되어 있습니다.

$$\sum_j \alpha_{i j}=1$$

Attention이라는 기법이 어떤 의미를 가지는지 정리가 조금 되셨나요? 우리는 Bahdanau의 기법을 조금 더 알고 싶으니 그에 포커싱한 멋진 시각화 글을 소개해드립니다. `2b. Luong et. al (2015) [2]` 부터는 지금 읽지 않으셔도 좋아요!

- [어텐션 (Attention)](https://modulabs.co.kr/blog/introducing-attention)

주의❗ 위 시각화 글의 `1 단계 : 모든 인코더 hidden state의 점수 얻기` 부분에서 두 Hidden State의 평가 함수에 내적을 사용했는데요. 일반적인 attention 설명을 위해 내적을 사용했을 뿐, Bahdanau attention에서 실제 평가 함수는 아래와 같이 **특정 벡터 공간으로 매핑된 두 Hidden State의 합**을 사용합니다.

![image.png](lms-img:0)

논문: [NEURAL MACHINE TRANSLATION BY JOINTLY LEARNING TO ALIGN AND TRANSLATE](https://arxiv.org/pdf/1409.0473.pdf)

> **[문제] 위 시각화 글의 4단계에서, hidden state에 softmax 값을 곱하여 alignment score를 얻었습니다. 해당 값들을 모두 더하여 최종적인 컨텍스트 벡터를 얻는데요, 다 더해지면 각 값들의 의미가 모호해지지 않을까요? 기존 Seq2Seq의 고정 크기의 컨텍스트 벡터와 비교하여 설명해봅시다. (Hint: Word2Vec의 연산을 기억하나요?)**

Bahdanau가 제안한 Attention은 하나의 Baseline이 되어 지금도 여러 기법을 시험하는 데에 멋진 중심점이 되어주고 있습니다. 멋진 기술들의 공통점은 구현한 것을 봤을 때도 이해가 정말 잘 된다는 것이죠! 여기까지의 개념설명이 명확히 와닿지 않았더라도, 구현을 살펴보며 복습을 해봅시다!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, units):
        super(BahdanauAttention, self).__init__()
        self.W_decoder = nn.Linear(hidden_dim, units)  # Decoder hidden state -> units
        self.W_encoder = nn.Linear(hidden_dim, units)  # Encoder hidden state -> units
        self.W_combine = nn.Linear(units, 1)           # Alignment score -> scalar weight

    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)          # (batch, seq_len, hidden_dim)

        W_enc = self.W_encoder(H_encoder)                       # (batch, seq_len, units)
        print("[ W_encoder X H_encoder ] Shape:", W_enc.shape)

        print("\n[ H_decoder ] Shape:", H_decoder.shape)        # (batch, hidden_dim)
        W_dec = self.W_decoder(H_decoder.unsqueeze(1))          # (batch, 1, units)
        print("[ W_decoder X H_decoder ] Shape:", W_dec.shape)

        score = self.W_combine(torch.tanh(W_dec + W_enc))       # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", score.shape)

        attention_weights = F.softmax(score, dim=1)             # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 가중합은 사영(W_encoder)하기 전의 원본 인코더 상태에 적용한다
        context_vector = torch.sum(attention_weights * H_encoder, dim=1)   # (batch, hidden_dim)
        print("\n[ Context Vector ] Shape:", context_vector.shape)

        return context_vector, attention_weights

# 설정
hidden_dim = 512
W_size = 100
print(f"Hidden State를 {W_size}차원으로 Mapping\n")

# 모델 생성
attention = BahdanauAttention(hidden_dim, W_size)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, hidden_dim))      # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

Hidden State를 100차원으로 Mapping

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 100])

[ H_decoder ] Shape: torch.Size([1, 512])
[ W_decoder X H_decoder ] Shape: torch.Size([1, 1, 100])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.09915758 0.08852635 0.10657677 0.09417299 0.09096252 0.11969227
  0.11380081 0.09530205 0.09322105 0.09858769]]

[ Context Vector ] Shape: torch.Size([1, 512])


Encoder의 모든 스텝에 대한 Hidden State를 100차원의 벡터 공간으로 매핑 `(1, 10, 100)` 하고, Decoder의 현재 스텝에 대한 Hidden State 역시 100차원의 벡터 공간으로 매핑 `(1, 1, 100)`해 **두 State의 합으로 정의된 Score** `(1, 10, 1)` **를 구하는 모습**입니다. Softmax를 거쳐 나온 값은 0-1 사이의 값으로 각 단어가 차지하는 비중을 의미하겠죠? 예시에서는 랜덤한 값을 사용했기 때문에 비중이 비슷비슷하지만 실제 단어로 적용시켜보면 **유사한 단어에 높은 비중을 할당**하게 된답니다! 그것을 시각화하면 아래와 같은 그림을 보실 수 있습니다.

![https://www.tensorflow.org/tutorials/text/nmt_with_attention?hl=ko](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/5e191cbc-952c-4938-bb2d-e048ec2922c2.png)

Attention Map

## (2) Luong Attention
---
Luong의 Attention은 Bahdanau의 방식을 약간 발전시킨 형태입니다. Decoder의 현재 Hidden State를 구하기 위해 한 스텝 이전의 Hidden State를 활용하는 것은 연산적으로 비효율적입니다. 이는 RNN의 연산 형태 때문인데, 자세한 내용은 아래 웹페이지에서 확인하시죠! 수식적인 부분은 완벽하게 이해하지 않아도 좋으니, Luong의 아이디어에 중점을 맞추도록 합니다.

- [[Attention] Luong Attention 개념 정리](https://hcnoh.github.io/2019-01-01-luong-attention)

정리 글을 읽어 보셨다면, 논문도 함께 첨부하니 멋진 아이디어가 인상 깊었다면 살펴보세요!

> [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/pdf/1508.04025.pdf)

> **[문제] 4가지 Score 함수(Dot, General, Concat, Location) 중, 가장 합리적인 성능을 보이는 함수는 어떤 것인가요?**

가장 좋은 성능을 보인 Score 함수는 아래와 같죠? 이를 기반으로 구현을 해보겠습니다.

$$\operatorname{Score}\left(H_{\text {target }}, H_{\text {source }}\right)=H_{\text {target }}^T \times W_{\text {combine }} \times H_{\text {source }}$$

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, units):
        super(LuongAttention, self).__init__()
        self.W_combine = nn.Linear(units, units)  # Encoder hidden state 변환

    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, hidden_dim)

        WH = self.W_combine(H_encoder)  # (batch, seq_len, hidden_dim)
        print("[ W_encoder X H_encoder ] Shape:", WH.shape)

        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        alignment = torch.bmm(WH, H_decoder.transpose(1, 2))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", alignment.shape)

        attention_weights = F.softmax(alignment, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        attention_weights = attention_weights.squeeze(-1)  # (batch, seq_len)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), H_encoder)  # (batch, 1, hidden_dim)
        context_vector = context_vector.squeeze(1)  # (batch, hidden_dim)

        return context_vector, attention_weights

# 설정
hidden_dim = 512
attention = LuongAttention(hidden_dim)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, hidden_dim))  # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 512])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.00432004 0.00981363 0.47364503 0.15567575 0.03677689 0.00347412
  0.04348961 0.08081377 0.04198393 0.15000728]]


Bahdanau의 Score 함수와는 다르게 하나의 Weight만을 사용하는 것이 특징입니다. 어떤 벡터 공간에 매핑해주는 과정이 없기 때문에 Weight의 크기는 인코더·디코더 hidden state 크기와 동일해야 연산이 가능합니다. 이 또한 번역에 적용해보고 성능을 비교해본다면 좋겠죠!

## 6. 트랜스포머로 가기 전 징검다리?

Seq2seq와 Attention이 폭풍처럼 휩쓸고 난 후, 잠잠해진 NLP 계를 다시 깨운 것은 2016년 구글의 신경망 번역 시스템이었습니다. 놀라운 구조를 제안한 것은 아니나 무려 8개 층을 쌓은 Encoder-Decoder 구조와 Residual Connection은 제법 멋졌죠. 이에 대한 정리 글을 첨부하니 가볍게 읽어 보세요!

- [Google's Neural Machine Translation System.](https://norman3.github.io/papers/docs/google_neural_machine_translation.html)

> **[문제] Residual Connection을 적극 활용했을 때의 이점이 몇 가지 떠오릅니다! 가장 인상 깊게 느껴지는 두 가지만 적어봅시다.**

GNMT(Google Neural Machine Translation)는 어쩌면 복선이었을 수도 있는데, 왜냐하면 그 후에 등장한 것이 NLP의 꽃, **트랜스포머(Transformer)** 이기 때문이죠! 앞서 언급한 레이어를 쌓는 구조나 Residual Connection이 트랜스포머와 굉장히 유사하기에 그렇게 느껴지기도 합니다.

![트랜스포머의 구조](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/cf742090-a90d-4ada-b32d-c6aaeb894bca.png)

트랜스포머 모델은 Multi-Head Attention이라는 개념을 도입해 폭넓은 문맥을 파악하게 하고, 기존의 RNN 구조를 완전히 탈피하여 연산 속도 측면에서도 혁신적인 발전이 일어났습니다! 지금까지도 트랜스포머를 기반으로 한 모델들이 각 분야에서 최고의 성능을 내고 있으니, 그 파급력을 알 만하죠?

궁금하시면 미리 알아보셔도 좋지만 오늘은 여기까지! 자세한 것은 차근차근 알아보도록 합시다.

## 7. 마무리하며

이번 코스에서는 언어 모델의 흐름과 Seq2seq, 그리고 이를 발전시키기 위한 Attention 기법을 두 가지 배웠습니다.
멋진 기술을 배운 만큼 어서 프로젝트를 해보고 싶은 마음이 굴뚝같으실 거라 믿어요 ^*^

![](https://resources-public-prd.modulabs.co.kr/home-section/story-modulabs-articles-section/cae3fe43-e716-43cf-9920-492b19598932.png)

## 종합퀴즈
---
지금까지 여러분들이 얼마나 학습을 충실히 하셨는지 알아볼까 합니다.
여러분의 실력을 쑥쑥 향상시켜줄 수 있는 퀴즈이기도 하므로 배운 내용을 다시 생각하면서 아래의 퀴즈를 풀어보세요. 🤗

> **[문제] RNN을 활용할 때 어떤 문제점을 주의해야할까요? 떠오르는대로 적어봅시다.**

> **[문제] Bahdanau, Luong attention 에 대해 나만의 언어로 간단하게 적어볼까요?**

종합 퀴즈는 괜찮았나요?
학습을 충실히 하셨다면 쉽게 해결하셨을 것이라 생각합니다.

아무리 완벽해 보이는 기술도 결점을 찾아내는 사람들이 있고, 결점이 곧 발전으로 이어지는 과정을 살펴보니 새삼 경이롭습니다. "역사는 반복된다"는 말이 있듯이, 이번 코스에서 우리가 배운 흐름들은 지금도 논문에 짠! 하고 등장하는 경우가 많습니다. 실제로 필자가 최근에 읽은 ELECTRA 모델 논문에 통계적 언어 모델이 등장해 "몰랐다면 이해하지 못했겠다..."고 생각한 경우도 있었으니까요.

그러니 너무 조급한 마음을 갖지 마시고, **천천히 이해하면서 진행하셔도 괜찮습니다**. 오히려 그게 멀리 갈 수 있는 방법이란 것은 다들 당연히 알고 계시겠죠? 차근차근 나아가서 멋진 연구자, 개발자, 과학자가 되시길 기원합니다! 화이팅!